# Registration and metrics with module

In [ ]:
import os 
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from skimage.transform import resize, AffineTransform, warp

from tifffile import imread

from __future__ import print_function

In [ ]:
import sys
sys.path.insert(0, r"D:\Masters_Medical_Informatics_Larger_Files\Thesis\Working_Folder\Masters-Thesis\src\registration")

In [ ]:
import registration
import importlib
importlib.reload(registration)
importlib.reload(registration.reg)
importlib.reload(registration.metrics)

#### Load data

In [ ]:
def load_image_data(folder_path):

    img_data = []
    label_data = []
    
    for _, _, files in os.walk(folder_path):
        for index, file in enumerate(files):
            if file.endswith(".tif"): 
                image_path = os.path.join(folder_path, file)
                img_raw = imread(image_path)
                img = np.array(img_raw)[:, :]
                img_data.append(img)

    return img_data

def display_image(img):
    img = img.astype(float)
    img = img / img.max()
    plt.imshow(img, cmap='gray', vmin=0, vmax=1)
    plt.colorbar()                 
    plt.show()

In [ ]:
scale = 0.5023/0.209877

# load dapi
dapi_img_data = load_image_data("../../../../Thesis_Data/for_valis/CRC027/cropped/dapi")
dapi1_init = resize(dapi_img_data[0], (int(dapi_img_data[0].shape[0]/scale), int(dapi_img_data[0].shape[1]/scale)), anti_aliasing=True)
dapi2_init = resize(dapi_img_data[1], (int(dapi_img_data[1].shape[0]/scale), int(dapi_img_data[1].shape[1]/scale)), anti_aliasing=True)

# load hne
hne_image_data = load_image_data("../../../../Thesis_Data/for_valis/CRC027/cropped/hne")
hne1_init = hne_image_data[0]
hne2_init = hne_image_data[1]

display_image(dapi1_init)
display_image(hne1_init)

#### Preprocess images

In [ ]:
hne1_deconv = registration.colour_deconvolusion_preprocessing_HnE(hne1_init)
hne2_deconv = registration.colour_deconvolusion_preprocessing_HnE(hne2_init)
display_image(hne1_deconv)
display_image(hne2_deconv)

#### Register images

In [ ]:
def overlay_images(dapi_img, hne_img, title='overlay'):
    h, w = dapi_img.shape
    plt.imshow(dapi_img, cmap='Greens', alpha=1)
    plt.imshow(hne_img, cmap='Reds', alpha=0.4)
    plt.title(title)
    plt.axis([0, w, h, 0])
    
    legend_patches = [
        Patch(color='green', label='DAPI', alpha=0.8),
        Patch(color='red', label='HnE', alpha=0.4)
    ]
    plt.legend(handles=legend_patches)

def display_img_comparison(dapi_img, hne_img, hne_deconv, registered_hne_imgs):
    no_comparisons = len(registered_hne_imgs)
    if 'intensity based' in registered_hne_imgs:
        no_comparisons += len(registered_hne_imgs['intensity based']) - 1

    h, w = dapi_img.shape
    
    plt.figure(figsize=((no_comparisons+3)*3, 6))

    plt.subplot(1, no_comparisons + 3, 1)
    plt.title('DAPI image')
    plt.imshow(dapi_img, cmap='gray')
    plt.axis([0, w, h, 0])

    plt.subplot(1, no_comparisons + 3, 2)
    plt.title('HnE image')
    plt.imshow(hne_img, cmap='gray')
    plt.axis([0, w, h, 0])

    plt.subplot(1, no_comparisons + 3, 3)
    plt.title('Initial Overlay')
    overlay_images(dapi_img, hne_deconv, title='initial overlay')

    for key, value in registered_hne_imgs.items():
        if key == 'intensity based':
            for key1, value1 in value.items():
                plt.subplot(1, no_comparisons + 3, list(value.keys()).index(key1) + 5)
                overlay_images(dapi_img, value1, title=key1)
        else:
            plt.subplot(1, no_comparisons + 3, list(registered_hne_imgs.keys()).index(key) + 4)
            overlay_images(dapi_img, value, title=key)

    plt.tight_layout()
    plt.show()

In [ ]:
# initial feature based similarity transform & advanced feature based projective transform
img1_tforms, img1_registered, img1_tre_pts = registration.register_DAPI_HnE(dapi1_init, hne1_deconv, adv_tform='feature', feature_tform='projective')
display_img_comparison(dapi1_init, hne1_init, hne1_deconv,img1_registered)
tre = registration.compute_TRE(img1_tforms, img1_tre_pts, dapi1_init)

In [ ]:
# initial feature based similarity transform & advanced feature based affine transform
img1_tforms, img1_registered, img1_tre_pts = registration.register_DAPI_HnE(dapi1_init, hne1_deconv, adv_tform='feature', feature_tform='affine')
display_img_comparison(dapi1_init, hne1_init, hne1_deconv,img1_registered)
tre = registration.compute_TRE(img1_tforms, img1_tre_pts, dapi1_init)

In [ ]:
# initial feature based similarity transform & follow-up intensity based rigid-affine-bspline transform
img1_tforms, img1_registered, img1_tre_pts = registration.register_DAPI_HnE(dapi1_init, hne1_deconv, adv_tform='intensity', intensity_tform='r-af-bs', mpp=0.5023)
display_img_comparison(dapi1_init, hne1_init, hne1_deconv,img1_registered)
tre = registration.compute_TRE(img1_tforms, img1_tre_pts, dapi1_init, mpp=0.5023)

In [ ]:
# initial feature based similarity transform 
img1_tforms, img1_registered, img1_tre_pts = registration.register_DAPI_HnE(dapi1_init, hne1_deconv)
display_img_comparison(dapi1_init, hne1_init, hne1_deconv,img1_registered)
tre = registration.compute_TRE(img1_tforms, img1_tre_pts, dapi1_init)

In [ ]:
# initial feature based similarity transform & advanced feature based affine transform
img2_tforms, img2_registered, img2_tre_pts = registration.register_DAPI_HnE(dapi2_init, hne2_deconv, adv_tform='feature', feature_tform='affine')
display_img_comparison(dapi2_init, hne2_init, hne2_deconv,img2_registered)
tre = registration.compute_TRE(img2_tforms, img2_tre_pts, dapi2_init)

In [ ]:
# initial feature based similarity transform & follow-up intensity based bspline transform
img2_tforms, img2_registered, img2_tre_pts = registration.register_DAPI_HnE(dapi2_init, hne2_deconv, adv_tform='intensity', intensity_tform='bspline', mpp=0.5023)
display_img_comparison(dapi2_init, hne2_init, hne2_deconv,img2_registered)
tre = registration.compute_TRE(img2_tforms, img2_tre_pts, dapi2_init, mpp=0.5023)

In [ ]:
# initial feature based similarity transform
hne1_applying_tform = AffineTransform(scale=(1.0, 1.0), rotation=0.5, translation=(100, -200))
hne1_testing_img = warp(hne1_deconv, hne1_applying_tform)
img1_test_tforms, img1_test_registered, img1_test_tre_pts = registration.register_DAPI_HnE(dapi1_init, hne1_testing_img, adv_tform='feature', feature_tform='affine')
display_img_comparison(dapi1_init, hne1_testing_img, hne1_deconv,img1_test_registered)

In [ ]:
import itk
from importlib import resources
from skimage.util import img_as_float32

transformation = 'bspline'
mpp = 0.5023
fixed = dapi1_init
moving = img1_registered['initial similarity']

if transformation == 'rigid':
    transform_scheme = ["01_Rigid"]
elif transformation == 'affine':
    transform_scheme = ["02_Affine"]
elif transformation == 'bspline':
    transform_scheme = ["03_BSpline"]
else:
    raise ValueError("transformation for intensity based registration must be one of 'rigid', 'affine', or 'bspline'")

global_trf_map = itk.ParameterObject.New()

parameter_maps = []
for tform in transform_scheme:
    with resources.path("registration.tform_params_adv", f"{tform}.txt") as p:
        parameter_maps.append(str(p))

fixed_itk = itk.GetImageFromArray(img_as_float32(fixed))
fixed_itk.SetSpacing([mpp,mpp])

moving_itk = itk.GetImageFromArray(img_as_float32(moving))
moving_itk.SetSpacing([mpp,mpp])

for Reg in parameter_maps:
    reg_map = itk.ParameterObject.New()
    reg_map.AddParameterFile(str(Reg))

    moving_itk, result_trf_params = itk.elastix_registration_method(
        fixed_itk, 
        moving_itk,
        parameter_object = reg_map,
        log_to_console=False
    )

    global_trf_map.AddParameterMap(result_trf_params.GetParameterMap(0))

no_of_transforms = global_trf_map.GetNumberOfParameterMaps()

transformed_img = []
result = moving_itk
for n in range(no_of_transforms):
    single_transform = itk.ParameterObject.New()
    single_transform.AddParameterMap(global_trf_map.GetParameterMap(n))
    result = itk.transformix_filter(result, single_transform, log_to_console=False)
    transformed_img.append(itk.GetArrayFromImage(result))

In [ ]:
img = np.zeros(dapi1_init.shape, dtype=np.float32)
img[100,100] = 1.0
moving_mask = itk.GetImageFromArray(img_as_float32(img))
moving_mask.SetSpacing([mpp,mpp])

global_trf_map.SetParameter("FinalBSplineInterpolationOrder", '0')

result_moving_mask = itk.transformix_filter(
    moving_mask, 
    global_trf_map,
    log_to_console=False
)

arr = itk.GetArrayFromImage(result_moving_mask)
y, x = np.where(arr > 0.5)
print(x, y)
print(len(x), len(y))
print(img.shape, arr.shape)
print(arr[100-5:100+5, 100-5:100+5])


## Testing

In [ ]:
from skimage.feature import SIFT, match_descriptors, plot_matched_features
from skimage import measure

def registration_with_SIFT(dapi_img, hne_img, max_ratio=0.6, n_octaves=3, n_scales=5):
    dapiX, dapiY = dapi_img.shape
    hneX, hneY = hne_img.shape
    scale_factor = 4

    # Resize the images to reduce memory usage
    dapi_scaled = resize(dapi_img, (dapiX // scale_factor, dapiY // scale_factor), anti_aliasing=True)
    hne_scaled = resize(hne_img, (hneX // scale_factor, hneY // scale_factor), anti_aliasing=True)

    descriptor_extractor = SIFT(n_octaves=n_octaves, n_scales=n_scales)

    descriptor_extractor.detect_and_extract(hne_scaled)
    keypoints1, descriptors1 = descriptor_extractor.keypoints, descriptor_extractor.descriptors

    descriptor_extractor.detect_and_extract(dapi_scaled)
    keypoints2, descriptors2 = descriptor_extractor.keypoints, descriptor_extractor.descriptors

    matches12 = match_descriptors(
        descriptors1, descriptors2, max_ratio=max_ratio, cross_check=True
    )

    # Extract matched keypoints
    src, dst = keypoints1[matches12[:, 0]], keypoints2[matches12[:, 1]]

    fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(11, 8))
    plt.gray()
    plot_matched_features(hne_scaled, dapi_scaled, keypoints0=keypoints1, keypoints1=keypoints2, matches=matches12, ax=ax[0])
    ax[0].axis('off')
    ax[0].set_title("H&E vs. DAPI\n" "(all keypoints and matches)")


    plot_matched_features(hne_scaled, dapi_scaled, keypoints0=keypoints1, keypoints1=keypoints2, matches=matches12, ax=ax[1], only_matches=True)
    ax[1].axis('off')
    ax[1].set_title("H&E vs. DAPI\n" "(subset of matches for visibility)")

    plt.tight_layout()
    plt.show()

    dst, src = dst * scale_factor, src * scale_factor

    # Compute affine transformation using RANSAC 
    model_robust, inliers = measure.ransac((dst, src),
                               AffineTransform, min_samples=4,
                               residual_threshold=2, max_trials=1000)
    
    hnetemp_matches = src[inliers] 
    dapitemp_matches = dst[inliers] 
    
    hne_matches = hnetemp_matches[:, [1, 0]].copy()
    dapi_matches = dapitemp_matches[:, [1, 0]].copy()

    return model_robust, [hne_matches, dapi_matches]

def overlay_images(dapi_img, hne_img, title='overlay'):
    plt.imshow(dapi_img, cmap='Greens', alpha=1)
    plt.imshow(hne_img, cmap='Reds', alpha=0.4)
    plt.title(title)
    plt.axis('off')
    
    legend_patches = [
        Patch(color='green', label='DAPI', alpha=0.8),
        Patch(color='red', label='HnE', alpha=0.4)
    ]
    plt.legend(handles=legend_patches)


def display_reg_results_with_features(dapi_img, hne_img, model, is_hne_rgb=True):

    registered_image = warp(hne_img, model.inverse, output_shape=dapi_img.shape)
    
    # Display results
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 3, 1)
    plt.title('HnE')
    plt.imshow(hne_img, cmap='gray')

    plt.subplot(1, 3, 2)
    plt.title('DAPI')
    plt.imshow(dapi_img, cmap='gray')

    plt.subplot(1, 3, 3)
    plt.title('Registered Image (aligned version of HnE)')
    plt.imshow(registered_image, cmap='gray')

    plt.show()

    # Display overlay
    plt.figure(figsize=(12,6))
    plt.subplot(1,2,1)
    overlay_images(dapi_img, 1-hne_img[...,0], title='Before Registration Overlay') if is_hne_rgb else overlay_images(dapi_img, hne_img, title='Before Registration Overlay')
    plt.subplot(1,2,2)
    overlay_images(dapi_img, 1-registered_image[...,0], title='After Registration Overlay') if is_hne_rgb else overlay_images(dapi_img, registered_image, title='After Registration Overlay')

    plt.show()

In [ ]:
model, _ = registration_with_SIFT(dapi1_init, img1_registered['initial similarity'])
display_reg_results_with_features(dapi1_init, img1_registered['initial similarity'], model, is_hne_rgb=False)
